# 104. Maximum Depth of Binary Tree
**Difficulty:** 🟢 Easy · **Topic:** Tree · **LeetCode:** https://leetcode.com/problems/maximum-depth-of-binary-tree/

## 💡 Concepts

**Core concept(s):** Walking the whole tree with **DFS (recursion)** or **BFS (a queue)**.

**Why it applies here:** Depth is just "how many levels tall". Either recurse (depth = 1 + the deeper child's depth) or peel the tree off level by level and count the levels.

**Key intuition:** A node's depth is one more than its taller child; keep asking each child the same question.

---

### 📚 What is a Binary Tree?
A **binary tree** is nodes in a branching shape: each node holds a value and up to two children (**left**, **right**). The top is the **root**; childless nodes are **leaves**; **height** is the longest root-to-leaf path.
- **In Python:** a small `TreeNode` class with `.val`, `.left`, `.right`.

### 📚 What is DFS (Depth-First Search) / Recursion?
**DFS** dives down one branch as far as possible, then backtracks. It's usually written with **recursion** — a function that calls itself on each child.
- **Complexity:** visits each node once → **O(n)** time; uses call-stack space up to the tree's **height**.

### 📚 What is BFS (Breadth-First Search) / a Queue?
**BFS** explores level by level using a **queue** (a first-in-first-out line): take a node, push its children to the back, repeat.
- **Complexity:** **O(n)** time; the queue can hold up to one full level.
- **In Python:** `collections.deque` (`append` to add, `popleft` to take from the front).

---

**Prerequisite knowledge:**
- A `TreeNode` class.
- Recursion, or a queue (`deque`).

## 📝 Problem

Return the maximum depth (number of nodes on the longest root-to-leaf path).

**Example**
```
    3
   / \
  9  20        -> depth 3
     / \
    15  7
```

> Two approaches with the **same** `O(n)` complexity — recursion vs a queue — shown to contrast the two core traversal styles.

In [1]:
from typing import Optional, List
from collections import deque

class TreeNode:
    """A single node of a binary tree: a value plus links to up to two children."""
    def __init__(self, val=0, left=None, right=None):
        self.val = val                     # the number stored at this node
        self.left = left                   # the left child (or None)
        self.right = right                 # the right child (or None)

def build_tree(values):
    """Build a tree from a level-order list, LeetCode style (None = missing child)."""
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0]); q = deque([root]); i = 1
    while q and i < len(values):
        node = q.popleft()                 # the parent we're attaching children to
        if i < len(values):                # attach the left child (if present)
            if values[i] is not None:
                node.left = TreeNode(values[i]); q.append(node.left)
            i += 1
        if i < len(values):                # attach the right child (if present)
            if values[i] is not None:
                node.right = TreeNode(values[i]); q.append(node.right)
            i += 1
    return root

def build_balanced(n):
    """Balanced BST holding 1..n (height ~log n) — used by the benchmark."""
    def helper(lo, hi):
        if lo > hi:
            return None
        mid = (lo + hi) // 2               # middle value becomes the subtree's root
        node = TreeNode(mid)
        node.left = helper(lo, mid - 1)    # smaller values go left
        node.right = helper(mid + 1, hi)   # larger values go right
        return node
    return helper(1, n)

def preorder(root):
    """Collect values in preorder: node, then left, then right."""
    out = []
    def go(n):
        if not n: return
        out.append(n.val); go(n.left); go(n.right)
    go(root); return out

def inorder(root):
    """Collect values in inorder: left, then node, then right (sorted for a BST)."""
    out = []
    def go(n):
        if not n: return
        go(n.left); out.append(n.val); go(n.right)
    go(root); return out

def same_shape(a, b):
    """True if two trees have identical shape and values."""
    if not a and not b: return True        # both empty -> match
    if not a or not b or a.val != b.val: return False  # one empty, or values differ
    return same_shape(a.left, b.left) and same_shape(a.right, b.right)

### Approach 1 — Recursion / DFS

**Idea:** An empty tree has depth 0. Otherwise depth = 1 + the larger of the two children's depths.

**Time complexity:** `O(n)` — every node once.

**Space complexity:** `O(h)` — recursion stack up to the tree's height.

In [ ]:
def max_depth_rec(root: Optional[TreeNode]) -> int:
    if not root:
        return 0                           # an empty tree has depth 0 (base case)
    # This node adds 1 on top of whichever child subtree is taller.
    return 1 + max(max_depth_rec(root.left), max_depth_rec(root.right))

### Approach 2 — BFS with a Queue

**Idea:** Process the tree level by level; count how many levels there are.

**Time complexity:** `O(n)`.

**Space complexity:** `O(n)` — the queue holds up to one full level.

In [ ]:
def max_depth_bfs(root: Optional[TreeNode]) -> int:
    if not root:
        return 0
    q = deque([root])                      # queue holds all nodes on the current level
    depth = 0
    while q:
        depth += 1                         # about to process one more level
        for _ in range(len(q)):            # snapshot: everything in q right now is one level
            node = q.popleft()
            if node.left:  
                q.append(node.left)   # queue up the next level's nodes
            if node.right: 
                q.append(node.right)
    return depth

In [ ]:
# Correctness check
tests = [([3,9,20,None,None,15,7],3), ([1,2],2), ([],0), ([1],1)]
for vals, exp in tests:
    root = build_tree(vals)
    a, b = max_depth_rec(root), max_depth_bfs(root)
    print(f"{vals} -> rec={a}, bfs={b} | expected={exp}")
    assert a == b == exp, "mismatch!"
print("\nAll tests passed")

## ⏱️ Empirically Checking the Complexities

Big-O can't be read off a function directly, but it can be **measured**. We time each approach on trees of growing size `n` and read the **doubling ratio** — how much runtime grows when `n` doubles.

| Theoretical | Ratio when `n` → `2n` |
|-------------|-----------------------|
| `O(log n)`    | ≈ **1×** |
| `O(n)`        | ≈ **2×** |
| `O(n²)`       | ≈ **4×** |

We use **balanced** trees (height ~log n) so deep recursion stays safe while every node is still visited.

In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "bench_utils.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
from bench_utils import benchmark

def make_worst_case(n):
    return (build_balanced(n),)
solutions = {
    "recursion O(n)": max_depth_rec,
    "bfs       O(n)": max_depth_bfs,
}
sizes = [1000, 2000, 4000, 8000]

benchmark(solutions, make_worst_case, sizes, plot=True)


## 🧩 Patterns Learned

- **Tree recursion:** "answer for a node = combine answers of its children" — the template for most tree problems.
- **DFS vs BFS:** recursion is shortest to write; a queue (BFS) avoids deep recursion and naturally groups levels.
- **Signal:** "depth / height / levels of a tree".
- **Related problems:** Balanced Binary Tree, Minimum Depth, Level Order Traversal.
- **Common pitfalls:** (1) forgetting the empty-tree base case; (2) confusing depth (nodes) with edges.